## Extract pathogenic variants from ClinVar data

I renamed the folder `clinvar`.  I then survey all *Clinical significance* categories in the data.

In [50]:
run kondrashov

In [3]:
vardata = pd.read_table('/Users/adesinamary/Documents/variant_summary.txt', sep='\t')

/var/folders/31/rr38lns17x78c7dt588fmqmw0000gs/T/ipykernel_80097/374571643.py:1: DtypeWarning: Columns (0: Chromosome) have mixed types. Specify dtype option on import or set low_memory=False.
  vardata = pd.read_table('/Users/adesinamary/Documents/variant_summary.txt', sep='\t')


In [ ]:
#vardata.ClinicalSignificance
vardata["GeneID"]
vardata.items()
vardata.keys()


Index(['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity',
       'SCVsForAggregateGermlineClassification',
       'SCVsForAggregateSomaticClinicalImpact',
       'SCVsForAggregateOncogenicityClassification'],
      dtype='str')

In [ ]:
signif = []
for gene in loci:
    vardata = pd.read_table('clinvar/' + gene + '_clinvar.txt.txt', sep='\t')
    n = len(vardata)
    for i in range(n):
        tmp = vardata['Clinical significance (Last reviewed)'][i]
        signif.append(tmp.split('(Last')[0])
    print(gene, n, len(signif))

ABCD1 325 325
ALPL 187 512
AR 189 701
ATP7B 509 1210
BTK 160 1370
CASR 770 2140
CBS 228 2368
CFTR 1289 3657
CYBB 151 3808
F7 50 3858
F8 363 4221
F9 181 4402
G6PD 119 4521
GALT 300 4821
GBA 160 4981
GJB1 415 5396
HBB 473 5869
HPRT1 56 5925
IL2RG 115 6040
KCNH2 1084 7124
KCNQ1 673 7797
L1CAM 148 7945
LDLR 1459 9404
MPZ 289 9693
MYH7 1625 11318
TYR 139 11457
PAH 602 12059
PMM2 109 12168
RHO 198 12366
TP53 1054 13420
TTR 147 13567
VWF 583 14150


In [3]:
categs = pd.Series(signif).value_counts()
categs

Uncertain significance                                       6400
Pathogenic                                                   2562
Likely pathogenic                                            1721
Conflicting interpretations of pathogenicity                 1178
not provided                                                  769
Pathogenic/Likely pathogenic                                  691
Likely benign                                                 285
other                                                         237
Benign                                                        126
Benign/Likely benign                                           65
Pathogenic, other                                              63
drug response                                                  16
no interpretation for the single variant                       12
Pathogenic, drug response                                      10
Pathogenic/Likely pathogenic, drug response                     6
Conflictin

We then construct a condition to isolate *Pathogenic* and *Likely pathogenic* variants only.

In [4]:
clin = categs.index.tolist()
for i in clin:
    if (i[:4]=='Path') or (i[:11]=='Likely path'):
        print(i)

Pathogenic
Likely pathogenic
Pathogenic/Likely pathogenic
Pathogenic, other
Pathogenic, drug response
Pathogenic/Likely pathogenic, drug response
Pathogenic/Likely pathogenic, other
Likely pathogenic, drug response
Pathogenic, risk factor
Pathogenic/Likely pathogenic, risk factor


The following categories are not included.

In [5]:
for i in clin:
    if (i[:4]=='Path') or (i[:11]=='Likely path'):
        pass
    else:
        print(i)

Uncertain significance
Conflicting interpretations of pathogenicity
not provided
Likely benign
other
Benign
Benign/Likely benign
drug response
no interpretation for the single variant
Conflicting interpretations of pathogenicity, other
Conflicting interpretations of pathogenicity, risk factor


Finally, we generate new spreadsheets with only pathogenic entries in a new folder `pathogenic`.  **We focus on those data from now on.**

In [2]:
!mkdir pathogenic

In [7]:
for gene in loci:
    pathog = []
    vardata = pd.read_table('clinvar/' + gene + '_clinvar.txt.txt', sep='\t')
    n = len(vardata)
    for i in range(n):
        signif = vardata['Clinical significance (Last reviewed)'][i]
        if (signif[:4]=='Path') or (signif[:11]=='Likely path'):
            pathog.append(True)
        else:
            pathog.append(False)
    vardata['Pathogenic'] = pathog
    print(gene, n, sum(pathog))
    tmp = vardata[vardata['Pathogenic']==True]
    del tmp['Pathogenic']
    tmp.to_csv('pathogenic/' + gene + '_clinvar.csv', index=False)

ABCD1 325 134
ALPL 187 76
AR 189 132
ATP7B 509 185
BTK 160 102
CASR 770 109
CBS 228 68
CFTR 1289 391
CYBB 151 55
F7 50 26
F8 363 269
F9 181 138
G6PD 119 53
GALT 300 174
GBA 160 103
GJB1 415 111
HBB 473 127
HPRT1 56 46
IL2RG 115 55
KCNH2 1084 206
KCNQ1 673 210
L1CAM 148 54
LDLR 1459 808
MPZ 289 97
MYH7 1625 270
TYR 139 72
PAH 602 358
PMM2 109 56
RHO 198 105
TP53 1054 241
TTR 147 77
VWF 583 149


# clinvar variants

### Download "variant_summary.txt" from Clinvar database [text](https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz)

In [ ]:
vardata = pd.read_table('/Users/adesinamary/Documents/variant_summary.txt', sep='\t')

In [24]:
!mkdir clinvar

mkdir: clinvar: File exists


In [51]:
terms = ["Pathogenic", "Likely pathogenic"]

XX = vardata[ 
(vardata["GeneSymbol"] == "ABCD1" )&
(vardata["ClinicalSignificance"].isin (terms)) &
(vardata["Type"] == "single nucleotide variant")
]

XX.to_csv(f"clinvar/ABCD1.txt", sep="\t", index= False)

In [49]:
len(XX)
XX
#print(XX["ClinicalSignificance"])

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
20553,26331,single nucleotide variant,NM_000033.4(ABCD1):c.871G>A (p.Glu291Lys),215,ABCD1,HGNC:61,Pathogenic,1,"Oct 24, 2024",128624213,...,A,-,-,-,-,-,-,SCV002023953,-,-
20554,26331,single nucleotide variant,NM_000033.4(ABCD1):c.871G>A (p.Glu291Lys),215,ABCD1,HGNC:61,Pathogenic,1,"Oct 24, 2024",128624213,...,A,-,-,-,-,-,-,SCV002023953,-,-
20555,26332,single nucleotide variant,NM_000033.4(ABCD1):c.1451C>G (p.Pro484Arg),215,ABCD1,HGNC:61,Pathogenic,1,"Dec 30, 1994",128624214,...,G,-,-,-,-,-,-,SCV000032279,-,-
20556,26332,single nucleotide variant,NM_000033.4(ABCD1):c.1451C>G (p.Pro484Arg),215,ABCD1,HGNC:61,Pathogenic,1,"Dec 30, 1994",128624214,...,G,-,-,-,-,-,-,SCV000032279,-,-
20557,26333,single nucleotide variant,NM_000033.4(ABCD1):c.1635-2A>G,215,ABCD1,HGNC:61,Pathogenic,1,"Feb 16, 2024",1569541109,...,G,-,-,-,-,-,-,SCV002122424|SCV002704504|SCV004702247,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8915686,4928921,single nucleotide variant,NM_000033.4(ABCD1):c.796G>C (p.Gly266Arg),215,ABCD1,HGNC:61,Likely pathogenic,1,"Dec 29, 2025",-1,...,C,-,-,-,-,-,-,SCV007528953,-,-
8977632,4960532,single nucleotide variant,NM_000033.4(ABCD1):c.274G>C (p.Gly92Arg),215,ABCD1,HGNC:61,Likely pathogenic,1,"May 26, 2025",-1,...,C,-,-,-,-,-,-,SCV007591950,-,-
8977633,4960532,single nucleotide variant,NM_000033.4(ABCD1):c.274G>C (p.Gly92Arg),215,ABCD1,HGNC:61,Likely pathogenic,1,"May 26, 2025",-1,...,C,-,-,-,-,-,-,SCV007591950,-,-
8980659,4962281,single nucleotide variant,NM_000033.4(ABCD1):c.1256T>C (p.Val419Ala),215,ABCD1,HGNC:61,Pathogenic,1,"May 12, 2026",-1,...,C,-,-,-,-,-,-,SCV007594731,-,-


### Test run with fewer genes

In [ ]:
loci = [
    "ABCD1",
    "ALPL",
    "AR",
    "ATP7B"]

terms = ["Pathogenic", "Likely pathogenic"]

for gene in loci:
    filtered = vardata[
        (vardata["GeneSymbol"] == gene )&
        (vardata["ClinicalSignificance"].isin (terms)) &
        (vardata["Type"] == "single nucleotide variant")
        ]
    filtered.to_csv(f"clinvar/{gene}.txt", sep="\t", index= False)
    print(f"{gene}: {len(filtered)} rows written")

# This code extracts variants of interest as .txt file, per gene, from the downloaded database

In [ ]:
run kondrashov

In [ ]:
terms = ["Pathogenic", "Likely pathogenic"]

for gene in loci:
    filtered = vardata[
        (vardata["GeneSymbol"] == gene )&
        (vardata["ClinicalSignificance"].isin (terms)) &
        (vardata["Type"] == "single nucleotide variant")
        ]
    filtered.to_csv(f"clinvar/{gene}.txt", sep="\t", index= False)
    print(f"{gene}: {len(filtered)} rows written")

ABCD1: 560 rows written
ALPL: 516 rows written
AR: 494 rows written
ATP7B: 562 rows written
BTK: 455 rows written
CASR: 336 rows written
CBS: 288 rows written
CFTR: 1248 rows written
CYBB: 276 rows written
F7: 122 rows written
F8: 949 rows written
F9: 520 rows written
G6PD: 332 rows written
GALT: 338 rows written
GBA1: 506 rows written
GJB1: 324 rows written
HBB: 296 rows written
HPRT1: 140 rows written
IL2RG: 216 rows written
KCNH2: 522 rows written
KCNQ1: 510 rows written
L1CAM: 256 rows written
LDLR: 1508 rows written
MPZ: 220 rows written
MYH7: 636 rows written
TYR: 342 rows written
PAH: 1300 rows written
PMM2: 232 rows written
RHO: 252 rows written
TP53: 490 rows written
TTR: 176 rows written
VWF: 570 rows written


In [ ]:
for gene in loci:
    pathog = []
    vardata = pd.read_table('clinvar/' + gene + '_clinvar_result.txt', sep='\t')
    n = len(vardata)
    for i in range(n):
        signif = vardata['Clinical significance (Last reviewed)'][i]
        if (signif[:4]=='Path') or (signif[:11]=='Likely path'):
            pathog.append(True)
        else:
            pathog.append(False)
    vardata['Pathogenic'] = pathog
    print(gene, n, sum(pathog))
    tmp = vardata[vardata['Pathogenic']==True]
    del tmp['Pathogenic']
    tmp.to_csv('pathogenic/' + gene + '_clinvar.csv', index=False)

In [ ]:
ls pathogenic/

ABCD1_clinvar.csv  CYBB_clinvar.csv   HBB_clinvar.csv    MYH7_clinvar.csv
ALPL_clinvar.csv   F7_clinvar.csv     HPRT1_clinvar.csv  PAH_clinvar.csv
AR_clinvar.csv     F8_clinvar.csv     IL2RG_clinvar.csv  PMM2_clinvar.csv
ATP7B_clinvar.csv  F9_clinvar.csv     KCNH2_clinvar.csv  RHO_clinvar.csv
BTK_clinvar.csv    G6PD_clinvar.csv   KCNQ1_clinvar.csv  TP53_clinvar.csv
CASR_clinvar.csv   GALT_clinvar.csv   L1CAM_clinvar.csv  TTR_clinvar.csv
CBS_clinvar.csv    GBA_clinvar.csv    LDLR_clinvar.csv   TYR_clinvar.csv
CFTR_clinvar.csv   GJB1_clinvar.csv   MPZ_clinvar.csv    VWF_clinvar.csv


In [3]:
!open .